# VLM cross-modal unlearning — Stage 1 runner

Thin runner: every cell calls one script; no experiment logic here. All outputs go to
`/content/drive/MyDrive/vlm_unlearn/`. Every job is **resumable**: after a disconnect, re-run the
setup cells (1a–1c) and then the job cell you were on. Order and runtimes: `README.md`.
Protocol: `PROTOCOL.md` + `DEVIATIONS.md`.

**Primary metric G** (`vlmu/metrics.py`): $G = \mathrm{Acc}_I^U - \mathrm{Acc}_T^U$, i.e. image+text accuracy minus
text-only accuracy of the unlearned model, on the frozen doubly-correct facts of the 30 confirmation
forget entities (primary forms, original option order). For image unlearning, $G_\text{reverse} = \mathrm{Acc}_T^U - \mathrm{Acc}_I^U$.

**Stop points.** Checkpoint A (cell 5): the vanilla answers the forget facts in *both* forms (gate ≥ 0.80 each).
Checkpoint B (cell 6b): at least one pilot checkpoint shows the leak. If either fails, stop and report.

**Order** (research.md, Section 6): 4 → 5 (A) → 6 → 6b pilot (B) → 6c logit-lens curves → *decision* → 7–9 grid.

**Starting model:** the repo's own vanilla fine-tune (both forms; LoRA on LM linear layers; vision tower and
projector frozen). `TODO(confirm)`: or `Kyomin/unlearning_target`.

## 1. Setup (run every session)

In [7]:
# --- set once ---------------------------------------------------------------
REPO_URL = "https://github.com/xuzhongwm/VLMUnlearn.git"
BRANCH = "main"
MODEL = "qwen2vl"   # "qwen2vl" = pipeline check only | "llava" = primary | "qwen25vl" = replication phase only
# ---------------------------------------------------------------------------
from google.colab import drive
drive.mount("/content/drive")
import os
os.environ["VLMU_ROOT"] = "/content/drive/MyDrive/vlm_unlearn"   # checkpoints, logs, results
os.environ["VLMU_LOCAL_CACHE"] = "/content/vlmu_cache"            # fast local copy of processed data
os.environ["HF_HOME"] = "/content/hf_home"                        # base model weights (local disk)
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.makedirs(os.environ["VLMU_ROOT"], exist_ok=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [8]:
import os
if not os.path.exists("/content/VLMUnlearn"):
    !git clone -b {BRANCH} {REPO_URL} /content/VLMUnlearn
else:
    !git -C /content/VLMUnlearn pull --ff-only
%cd /content/VLMUnlearn
!git log --oneline -1

Cloning into '/content/VLMUnlearn'...
remote: Enumerating objects: 217, done.
remote: Counting objects: 100% (217/217), done.
remote: Compressing objects: 100% (89/89), done.
remote: Total 217 (delta 124), reused 214 (delta 124), pack-reused 0 (from 0)
Receiving objects: 100% (217/217), 243.76 KiB | 2.94 MiB/s, done.
Resolving deltas: 100% (124/124), done.
/content/VLMUnlearn
c38ce49 (HEAD -> main, origin/main, origin/HEAD) Clean outdated files


In [9]:
!pip install -q -r requirements.txt
!nvidia-smi --query-gpu=name,memory.total --format=csv
!python -c "import torch, transformers, peft; print(torch.__version__, transformers.__version__, peft.__version__, torch.cuda.is_available())"

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.30.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.42.0 requires huggingface-hub<3.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompatible.
name, memory.total [MiB]
NVIDIA A100-SXM4-40GB, 40960 MiB
I0000 00:00:1793314171.517680    2057 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1793314171.587186    2057 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in perf

## 2. Data (once; CPU is fine) — ~10 min
Downloads MLLMU-Bench + MMBench to Drive, builds the frozen splits and paired items, runs the data checks.

In [10]:
!python scripts/prepare_data.py --config configs/data.yaml

[2026-10-29T22:50:01] downloading MLLMMU/MLLMU-Bench/ft_Data/train-00000-of-00001.parquet
ft_Data/train-00000-of-00001.parquet: 100% 238M/238M [00:05<00:00, 41.4MB/s] 
[2026-10-29T22:50:08] downloading MLLMMU/MLLMU-Bench/Full_Set/train-00000-of-00001.parquet
Full_Set/train-00000-of-00001.parquet: 100% 238M/238M [00:03<00:00, 78.1MB/s] 
[2026-10-29T22:50:12] downloading MLLMMU/MLLMU-Bench/Retain_Set/train-00000-of-00001.parquet
Retain_Set/train-00000-of-00001.parquet: 100% 1.58M/1.58M [00:00<00:00, 2.27MB/s]
[2026-10-29T22:50:13] downloading MLLMMU/MLLMU-Bench/forget_10/train-00000-of-00001.parquet
forget_10/train-00000-of-00001.parquet: 100% 23.7M/23.7M [00:00<00:00, 30.4MB/s]
[2026-10-29T22:50:14] downloading MLLMMU/MLLMU-Bench/Test_Set/train-00000-of-00002.parquet
Test_Set/train-00000-of-00002.parquet: 100% 337M/337M [00:14<00:00, 23.7MB/s]
[2026-10-29T22:50:29] downloading MLLMMU/MLLMU-Bench/Test_Set/train-00001-of-00002.parquet
Test_Set/train-00001-of-00002.parquet: 100% 336M/336M 

Optional: regenerate the E0 data audit (`analysis/data_audit.md`) from the Drive copy of the raw data.

In [11]:
!python scripts/data_audit.py

wrote analysis/data_audit.md and analysis/official_mc_attributes.csv (6521 questions)


Paired-item check: a random sample of facts in both forms (image + "the pictured person" vs name, no image) and of the rewritten FT QA. Read them once.

In [ ]:
!python scripts/show_pairs.py --n 8

## 3. Smoke test (once per model) — ~10–15 min
Runs every job below with `--debug` (~20 samples each) into `vlm_unlearn/debug/`; earlier debug outputs are archived, not deleted. The final sanity step is *expected* to report a failed gate (the debug vanilla trains for 6 steps). **Send me the output of this cell before running anything below.**

In [12]:
!python scripts/smoke_test.py --model {MODEL}

[2026-10-29T22:51:25] reading parquet files
[2026-10-29T22:51:33] build done: {"splits": {"forget": 4, "forget_calibration": 2, "forget_confirmation": 2, "retain_unlearn": 10, "retain_calibration": 2, "retain_confirmation": 2, "celeb": 4, "no_official_mc": 1}, "train_vanilla": {"img": 283, "txt": 265}, "unlearn_forget": {"img": 61, "txt": 61}, "unlearn_retain": {"img": 148, "txt": 148}, "text_rewrite_failures": 0, "items_forget": {"paired/img": 36, "paired/txt": 36, "paired/pose": 36, "official/img": 20, "official/txt": 20}, "items_retain": {"paired/img": 36, "paired/txt": 36, "paired/pose": 36, "official/img": 20, "official/txt": 20}, "items_celeb": {"official/img": 20, "official/txt": 20}, "items_utility": {"mmbench/img": 20}, "dropped_by_reason": {}}
[2026-10-29T22:51:33] PASS splits pairwise disjoint
[2026-10-29T22:51:33] PASS forget = calibration + confirmation (all unlearned)
[2026-10-29T22:51:33] PASS unlearn_retain profiles within retain_unlearn
[2026-10-29T22:51:33] PASS paire

---
**Cells 4–9 are the real runs. Do not start them until the protocol owner approves.**

## 4. Pretrained baseline (no fine-tuning: measures priors)

In [ ]:
!python scripts/eval_pretrained.py --config configs/{MODEL}/vanilla.yaml

## 5. Vanilla fine-tune (both forms) + full eval + **checkpoint A** + frozen doubly-correct set
Checkpoint A = the gate: paired forget MC accuracy ≥ 0.80 in **both** image+text and text-only form. MLLMU-Bench's
own FT data is image QA only, so the text form is the one at risk. Compare the two numbers in `gate.json`:
if the gate fails (or the text form is far below the image form), unlearning is blocked: stop and report.

In [ ]:
!python scripts/finetune_vanilla.py --config configs/{MODEL}/vanilla.yaml
!cat /content/drive/MyDrive/vlm_unlearn/runs/{MODEL}/vanilla/gate.json

[2026-10-29T22:55:04] copying /content/drive/MyDrive/vlm_unlearn/data/processed -> /content/vlmu_cache/content/drive/MyDrive/vlm_unlearn/data/processed (local cache)
I0000 00:00:1793314508.945031    3917 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1793314509.013358    3917 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1793314511.358178    3917 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the envir

## 6. Milestone J on the real model: hook checks (i)–(iii) and tool tests J1–J6, a few minutes
Runs in float32 (bf16 rounding made (ii)/(iii-b) fail in the first smoke test; see DEVIATIONS.md DEV-007).
J6 logs the cross-route patched tokens of one pair: check them by eye once.

In [ ]:
!python scripts/check_hooks.py --config configs/{MODEL}/grid.yaml

Optional: the same checks as unit tests (`-s` shows the J6 printout).

In [ ]:
!VLMU_TEST_CONFIG=configs/{MODEL}/grid.yaml python -m pytest -q -s tests/test_milestone_j.py

## 6b. Pilot: one cell (text-only unlearning, NPO + retain NLL, one lr, one seed) + **checkpoint B**
Calibration checkpoints every 0.25 epoch; each saves its LoRA and MC scores; epoch boundaries add the open-ended
scores (teacher-forced answer log-prob, greedy keyword hit, ROUGE-L recall). Checkpoint B passes if some
checkpoint has: text forget accuracy ≤ 0.40, image minus text ≥ 0.10, retain drop ≤ 0.10 in both forms
(`configs/{MODEL}/pilot.yaml`, `TODO(confirm)` thresholds). **If it fails, stop and report; do not run the grid.**

In [ ]:
!python scripts/pilot.py --config configs/{MODEL}/pilot.yaml
!cat /content/drive/MyDrive/vlm_unlearn/results/{MODEL}/pilot_summary.json

## 6c. Logit-lens curves (`layer_logit_diff`) on the first checkpoint that passed B
Per layer: log p(correct letter), unlearned minus vanilla, for the image and text route.
**Decision (by a person):** image curve dips then recovers → ablation of the recovery layers (H2);
image curve stays ≈ 0 → `lora_layer_mask` + `delta_wx_norm` (H1 vs H1′). The full grid (7–9) comes after this.

In [ ]:
!python scripts/mech_layer_logit_diff.py --config configs/{MODEL}/pilot.yaml
import glob
from IPython.display import Image, display
for f in sorted(glob.glob(f"/content/drive/MyDrive/vlm_unlearn/results/{MODEL}/mech/layer_logit_diff_step*.png")):
    display(Image(f))

## 7. Unlearning grid (GA_Diff, NPO × text/image unlearning × 3 lr × 2 seeds = 24 runs): only after the 6c decision
Each run stops at its earliest eligible calibration checkpoint. Skips finished runs; re-run after a disconnect. Failed runs are logged in `results/run_log.csv` and the grid continues.

In [ ]:
!python scripts/run_grid.py --config configs/{MODEL}/grid.yaml

## 8. Selection (calibration only) + full evals of the selected checkpoints (confirmation entities)

In [ ]:
!python scripts/select_and_eval.py --config configs/{MODEL}/grid.yaml

## 9. Tables, figure, verdict + sanity checks

In [ ]:
!python scripts/aggregate.py --config configs/{MODEL}/grid.yaml
!python scripts/sanity.py --config configs/{MODEL}/grid.yaml
!cat /content/drive/MyDrive/vlm_unlearn/results/{MODEL}/verdict.json

## 10. (optional) push results to the repo

In [ ]:
!git add results analysis && git -c user.email=colab@local -c user.name=colab commit -m "results: {MODEL}" && git push